In [ ]:
import haiku as hk
import jax
import jax.numpy as jnp
import pandas as pd
import pickle
import sys

from pathlib import Path

# Get the absolute path to the root of the repository, to use the models' python files
repo_root = Path.cwd().parent.parent  
sys.path.append(str(repo_root))

from BulkRNABert.multiomics_open_research.bulk_rna_bert.downstream.pretrained import get_pretrained_downstream_model
from BulkRNABert.multiomics_open_research.common.preprocess import preprocess_omic

In [ ]:
# we need python 3.11 to run the pretrained model BulkRNABert
from platform import python_version
python_version()

'3.11.14'

In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax

parameters, forward_fn, tokenizer, config, mlm_config = get_pretrained_downstream_model(
    model_name="tcga_5_cohorts",
    checkpoint_directory="../checkpoints/",
)
forward_fn = hk.transform(forward_fn)

# Get bulk RNASeq data and tokenize it
rna_seq_df = pd.read_csv("../data/bulkrnabert/tcga_sample.csv")
rna_seq_array = preprocess_omic(rna_seq_df, mlm_config)
tokens_ids = tokenizer.batch_tokenize(rna_seq_array)
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

# The pre-trained model outputs (batch, 5) logits
# The features before the final layer are (batch, 128)
# We can extract them by removing the final linear layer

def create_regression_model():
    """Create a model that uses pre-trained embeddings + new regression head"""
    
    def regression_forward(tokens):
        # Call original model
        output = forward_fn.apply(parameters, jax.random.PRNGKey(0), tokens)
        
        # Extract logits: shape (batch, 5)
        logits = output['logits']
        
        # The 128-dim features are computed internally
        # We can approximate by using the logits as input to a regression layer
        # OR: redefine the model to return intermediate features
        
        # For simplicity: use a learned transformation from 5 → 1
        # (This is OK if you want to keep all pre-trained weights)
        survival = hk.Linear(1, name="regression_head")(logits)
        return survival
    
    return regression_forward

# Transform the model
regression_model = hk.transform(create_regression_model())

# Initialize new parameters (only the regression head)
rng = jax.random.PRNGKey(0)
new_params = regression_model.init(rng, tokens[:1])

# All parameters (pre-trained + new regression head)
all_params = {**parameters, **new_params}

# Training function
def loss_fn(params, tokens, survival_labels):
    """MSE loss for regression"""
    predictions = regression_model.apply(params, rng, tokens)  # (batch, 1)
    loss = jnp.mean((predictions - survival_labels.reshape(-1, 1)) ** 2)
    return loss

# Training step
optimizer = optax.adam(learning_rate=1e-4)
opt_state = optimizer.init(all_params)

def train_step(params, opt_state, tokens, survival_labels):
    loss, grads = jax.value_and_grad(loss_fn)(params, tokens, survival_labels)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = jax.tree_util.tree_map(lambda p, u: p + u, params, updates)
    return params, opt_state, loss

# Example usage
batch_tokens = tokens[:10]  # 10 samples
batch_survival = jnp.array([365, 500, 200, 600, 450, 300, 700, 400, 550, 380])  # days

for epoch in range(10):
    params, opt_state, loss = train_step(params, opt_state, batch_tokens, batch_survival)
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Inference
survival_pred = regression_model.apply(params, rng, tokens[:1])
print(f"Predicted survival (days): {survival_pred[0, 0]}")

In [ ]:
# /!\ does not work with the latest version of transformers, which added new parameters to the config that are not in the model's code, and thus cannot be loaded. 
# Load model and tokenizer
# config = AutoConfig.from_pretrained("InstaDeepAI/BulkRNABert", trust_remote_code=True)
# config.embeddings_layers_to_save = (4,)  # last transformer layer

# Try loading with ignore of unknown params
from transformers.configuration_utils import PreTrainedConfig
try:
    config = AutoConfig.from_pretrained(
        "InstaDeepAI/BulkRNABert", 
        trust_remote_code=True
    )
except TypeError as e:
    if "attention_maps_to_save" in str(e):
        # Load config and manually remove the problematic param
        
        config_file = hf_hub_download(
            repo_id="InstaDeepAI/BulkRNABert",
            filename="config.json",
            repo_type="model"
        )
        
        with open(config_file) as f:
            config_dict = json.load(f)
        
        # Remove problematic parameters
        config_dict.pop("attention_maps_to_save", None)
        config_dict.pop("embeddings_layers_to_save", None)
        
        # Load config from dict
        config = PreTrainedConfig.from_dict(config_dict)
    else:
        raise

tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/BulkRNABert", 
    config=config,  # Pass the cleaned config here
    trust_remote_code=True
)
model = AutoModel.from_pretrained("InstaDeepAI/BulkRNABert", config=config, trust_remote_code=True)

# Load bulk RNA-seq data and preprocess them.
csv_path = hf_hub_download(
    repo_id="InstaDeepAI/BulkRNABert",
    filename="data/tcga_sample.csv",
    repo_type="model",
)
gene_expression_array = pd.read_csv(csv_path).drop(["identifier"], axis=1).to_numpy()[:1, :]
gene_expression_array = np.log10(1 + gene_expression_array)
assert gene_expression_array.shape[1] == config.n_genes

# Tokenize
gene_expression_ids = tokenizer.batch_encode_plus(gene_expression_array, return_tensors="pt")["input_ids"]

# Compute BulkRNABert's embeddings
gene_expression_mean_embeddings = model(gene_expression_ids)["embeddings_4"].mean(axis=1)  # embeddings can be used for downstream tasks.